# Day 38 — Practice Questions
### Ask the Neighbours — k-Nearest Neighbours

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 6 |
| **B** | Find the mistake | 3 |
| **C** | Write the code | 4 |
| **D** | Challenge | 1 |

### Three questions to keep asking

- Are the columns **scaled**? If not, nothing else matters.
- Where did **k** come from — cross-validation, or a guess?
- How many **columns** are there, and do they all carry signal?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV, train_test_split)
from sklearn.metrics import accuracy_score, confusion_matrix

import warnings; warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

### A1 · Distance by hand

Three stored patients, and a new one.

| | temperature | heart rate | ill? |
|---|---|---|---|
| **P** | 38.0 | 95 | yes |
| **Q** | 36.5 | 72 | no |
| **R** | 37.2 | 88 | yes |
| **new** | 37.0 | 80 | ? |

Work out the distance from the new patient to each of P, Q and R, and say what
`k = 1` predicts.

In [2]:
pts = {"P": (38.0, 95, "yes"), "Q": (36.5, 72, "no"), "R": (37.2, 88, "yes")}
new = (37.0, 80)
for nm, (t, h, lab) in pts.items():
    d = np.sqrt((t - new[0])**2 + (h - new[1])**2)
    print(f"{nm}: sqrt(({t}-37.0)² + ({h}-80)²) = sqrt({(t-new[0])**2:.2f} + {(h-new[1])**2:.0f})"
          f" = {d:.4f}   ({lab})")

P: sqrt((38.0-37.0)² + (95-80)²) = sqrt(1.00 + 225) = 15.0333   (yes)
Q: sqrt((36.5-37.0)² + (72-80)²) = sqrt(0.25 + 64) = 8.0156   (no)
R: sqrt((37.2-37.0)² + (88-80)²) = sqrt(0.04 + 64) = 8.0025   (yes)


### A2 · The same three, scaled

Suppose temperature has a standard deviation of **0.8** and heart rate **12**.
Scale all four points (subtract the mean, divide by the sd — use means of 37.0 and 85),
then redo A1.

In [3]:
mt, st, mh, sh = 37.0, 0.8, 85.0, 12.0
newz = ((37.0 - mt)/st, (80 - mh)/sh)
print(f"new scaled: ({newz[0]:+.4f}, {newz[1]:+.4f})\n")
for nm, (t, h, lab) in pts.items():
    tz, hz = (t - mt)/st, (h - mh)/sh
    d = np.sqrt((tz - newz[0])**2 + (hz - newz[1])**2)
    print(f"{nm}: scaled ({tz:+.4f}, {hz:+.4f})   distance {d:.4f}   ({lab})")

new scaled: (+0.0000, -0.4167)

P: scaled (+1.2500, +0.8333)   distance 1.7678   (yes)
Q: scaled (-0.6250, -1.0833)   distance 0.9138   (no)
R: scaled (+0.2500, +0.2500)   distance 0.7120   (yes)


### A3 · What does `.fit()` do?

Explain in one sentence each:

1. What `KNeighborsClassifier(15).fit(X, y)` actually does
2. Why KNN is sometimes called a "lazy" learner
3. Why its memory use grows with the training data and logistic regression's does not

### A4 · Pick k

For each, say what you would do and why.

1. You have 30 rows.
2. Two classes, and you are deciding between k = 10 and k = 11.
3. Someone picked k = 1 "because it gave 100% accuracy".
4. Your dataset has 10,000 rows. Is k = 2,500 reasonable?

### A5 · Why is KNN slow?

A model stores **50,000** rows with **8** columns. You ask it to predict **1,000** new
rows, with k = 15.

Roughly how many distance calculations is that? And what single thing would make the
biggest difference?

In [4]:
stored, cols, queries = 50_000, 8, 1_000
print(f"distances to compute: {queries:,} × {stored:,} = {queries*stored:,}")
print(f"each one touches {cols} columns -> about {queries*stored*cols:,} subtractions")

distances to compute: 1,000 × 50,000 = 50,000,000
each one touches 8 columns -> about 400,000,000 subtractions


### A6 · Would KNN beat logistic regression?

For each boundary, say which model you would expect to win.

1. Approve a loan if `income > 40,000`
2. Flag a machine if its temperature is **either below 70°C or above 95°C**
3. Predict rain from 400 weather-station readings
4. Two classes arranged as a bullseye — one class in a ring around the other

---
# Part B — Find the mistake

Three short programs. All run. All are wrong.

In [5]:
# ---------------------------------------------------------------
# 800 machines. A machine fails if it runs too cold OR too hot.
# B and C both use `mach`.
# ---------------------------------------------------------------
r = np.random.default_rng(3820)
N = 800

temperature_c = r.uniform(40, 120, N).round(1)
vibration_hz  = r.gamma(2.4, 1.3, N).clip(0.2, 12).round(2)
runtime_hours = r.uniform(0, 20000, N).round(0)

off = np.abs(temperature_c - 82.0) / 20.0
z = (-4.9 + 1.6*(1.25*off**2 + 0.42*vibration_hz + 0.000035*runtime_hours)
     + r.normal(0, 0.35, N))
failed = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

mach = pd.DataFrame({"temperature_c": temperature_c, "vibration_hz": vibration_hz,
                     "runtime_hours": runtime_hours, "failed": failed})
X, y = mach[["temperature_c", "vibration_hz", "runtime_hours"]], mach.failed
cv = StratifiedKFold(5, shuffle=True, random_state=0)

print(f"rows {N}   failed {int(y.sum())}   rate {y.mean():.4f}")
mach.head()

rows 800   failed 403   rate 0.5038


,temperature_c,vibration_hz,runtime_hours,failed
0,70.5,1.61,15907.0,0
1,98.8,4.44,2788.0,1
2,99.6,2.58,15570.0,1
3,67.8,6.14,19302.0,1
4,81.9,1.31,8371.0,0


### B1 · "KNN just does not work on this data" 

In [6]:
score = cross_val_score(KNeighborsClassifier(n_neighbors=15), X, y, cv=cv).mean()
print(f"KNN accuracy: {score:.4f}")
print()
print("REPORT: 'We tried KNN and it performs at chance. Recommend")
print("         we use logistic regression instead.'")

KNN accuracy: 0.5150

REPORT: 'We tried KNN and it performs at chance. Recommend
         we use logistic regression instead.'


### B2 · "We tuned k properly" 

In [7]:
best_k, best_score = None, 0
for k in range(1, 52, 2):
    s = make_pipeline(StandardScaler(), KNeighborsClassifier(k)).fit(X, y).score(X, y)
    if s > best_score:
        best_k, best_score = k, s
print(f"best k = {best_k}  with accuracy {best_score:.4f}")

best k = 1  with accuracy 1.0000


### B3 · "Scaling first, then cross-validating" 

In [8]:
X_scaled = StandardScaler().fit_transform(X)
score = cross_val_score(KNeighborsClassifier(15), X_scaled, y, cv=cv).mean()
print(f"accuracy: {score:.4f}")

accuracy: 0.8087


---
# Part C — Write the code

Four tasks on the machine data.

### C1 · Look before you model

Print the failure rate and the baseline. Then print the failure rate in temperature
bands of 40–60, 60–75, 75–90, 90–105 and 105–120 °C. What shape do you see, and what
does it tell you about which model will win?

In [9]:
# your code here

### C2 · Scale it, and prove it mattered

Cross-validate KNN with k = 15 unscaled and scaled, and add logistic regression and the
baseline for comparison.

In [10]:
# your code here

### C3 · Tune k properly

Use `GridSearchCV` to find the best k from 1 to 51 (odd values only), inside a Pipeline.
Report the best k and its score, and plot the whole curve.

In [11]:
# your code here

### C4 · Does `weights="distance"` help? Does the scaler matter?

Compare, at the best k from C3:

- `weights="uniform"` (the default) against `weights="distance"`
- `StandardScaler` against `MinMaxScaler`

Report all four, and say whether any difference is real.

In [12]:
# your code here

---
# Part D — Challenge

### D1 · Write KNN yourself

Write `my_knn(X_train, y_train, X_new, k)` using **no** scikit-learn. It should:

- work for any number of columns
- compute the distance from each new row to every training row
- take the k nearest and return the majority vote
- handle ties sensibly, and say what rule you chose

Then show it matches `KNeighborsClassifier` exactly on the machine data — **remember to
scale**.

In [13]:
# your function here

---
## Before you hand it in

- Is there a scaler, **inside a Pipeline**?
- Did k come from `GridSearchCV` or `cross_val_score` — never from `.score(X_train, y_train)`?
- Is k odd, for two classes?
- Did you report the **baseline** next to the accuracy?
- How many columns are there? Would a linear model have done better?

Four numbers worth carrying away:

- Unscaled KNN on the machine data scored **0.5150** — chance. Scaled, **0.8125**.
- Deleting one uninformative column (`runtime_hours`) lifted the *unscaled* model from
  0.5150 to **0.8025**, because its units were drowning the other two.
- `.fit(X, y).score(X, y)` returns **exactly 1.0000** at k = 1, on every dataset, always.
- Scaler choice and distance weighting moved the result by **0.0062** — less than a
  quarter of the fold-to-fold noise. Not everything is worth tuning.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*